# Attention — Hands-on Tutorial

In this notebook you will:
- Implement scaled dot-product attention on a tiny synthetic sequence and watch the QKV machinery
- Visualise an attention matrix — which token attends to which
- Patch-embed a synthetic galaxy image (the ViT trick) and look at where each patch attends
- Implement `scaled_dot_product_attention(Q, K, V)` from scratch and check against PyTorch

**Prior:** Notebooks 01–05 complete; slide deck 05a (attention / Vision Transformers)

## Where this fits

| # | Topic | Slide deck | Notebook |
|---|---|---|---|
| 1 | Single neuron | `01_single_neuron.pdf` | `01_single_neuron.ipynb` |
| 2 | Multilayer networks | `02_multilayer_networks.pdf` | `02_training_loop.ipynb` |
| 3 | Backpropagation | `03_backprop_training.pdf` | `03_backpropagation.ipynb` |
| 4 | Optimizers | `07_optimizers.pdf` *(new)* | `04_optimizers.ipynb` |
| 5 | CNNs | `04_cnns.pdf` | `05_cnns.ipynb` |
| **→ 6** | **Modern architectures** | **`05a_attention.pdf`** + `05b_practical.pdf` | **`05b_attention.ipynb`** + *(bonus: `bonus_generative_models.ipynb`)* |
| 7 | Bayesian inference | `06_bayesian_inference.pdf` | `06_bayesian_inference.ipynb` |

**Coming from:** Notebook 05 (CNNs) showed local-pattern matching at fixed positions. Attention lets every position look at every other position.

**Leading to:** Notebook 06 (Bayesian) closes the structured course; the bonus generative-models notebook is open-ended exploration.

**If you skipped ahead:** You need `nn.Linear`, the four-step training loop, and ideally a working understanding of CNNs.

In [ ]:
# ── Environment setup ─────────────────────────────────────────────
# Run this cell only on Colab. On JupyterHub (pixi env), packages are pre-installed.
import sys
if 'google.colab' in sys.modules:
    %pip install -q torch matplotlib

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

%matplotlib inline
torch.manual_seed(0)
np.random.seed(0)

---

## The mathematics of attention

### Scaled dot-product attention

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

Each input token $x_i \in \mathbb{R}^{d}$ is linearly projected three times:

$$Q = X W_Q, \qquad K = X W_K, \qquad V = X W_V$$

| Symbol | Shape | Meaning |
|--------|-------|---------|
| $X$ | $(N, d)$ | Input sequence of $N$ tokens, each of dimension $d$ |
| $W_Q, W_K, W_V$ | $(d, d_k)$ | Learned projection matrices |
| $Q$ | $(N, d_k)$ | Queries — “what am I looking for?” |
| $K$ | $(N, d_k)$ | Keys — “what do I contain?” |
| $V$ | $(N, d_k)$ | Values — “what do I contribute if attended to?” |
| $Q K^\top$ | $(N, N)$ | Raw attention scores — pairwise dot products |
| $\sqrt{d_k}$ | scalar | Scaling to keep softmax in a numerically friendly regime |

The output has shape $(N, d_k)$: each row is a *weighted average* of value vectors,
with weights given by query–key similarity. Unlike a CNN, every output position can pull
information from every input position in a single layer.

---

## Part 1 — Self-attention on a toy sequence

Eight random “tokens” in a 16-dimensional embedding space. We build $Q$, $K$, $V$ by
applying three random linear projections, compute the $8 \times 8$ attention matrix,
and visualise it.

Read the attention matrix as: *row $i$, column $j$* = how much token $i$ attends to token $j$.
Rows sum to 1 (softmax).

In [ ]:
# ── Part 1: Self-attention on a toy 8-token sequence ───────────────────────

N, d, d_k = 8, 16, 16     # sequence length, embedding dim, key/query dim

X = torch.randn(N, d)     # 8 random tokens in 16-D space

# Three learned projections (here: random, untrained — just to see the machinery)
W_Q = torch.randn(d, d_k) / np.sqrt(d)
W_K = torch.randn(d, d_k) / np.sqrt(d)
W_V = torch.randn(d, d_k) / np.sqrt(d)

Q = X @ W_Q       # (N, d_k)
K = X @ W_K       # (N, d_k)
V = X @ W_V       # (N, d_k)

# Scaled dot-product attention, by hand
scores  = Q @ K.T / np.sqrt(d_k)        # (N, N) raw scores
weights = F.softmax(scores, dim=-1)     # (N, N) attention weights, rows sum to 1
output  = weights @ V                   # (N, d_k)

print(f'X shape:       {tuple(X.shape)}')
print(f'Q, K, V shape: {tuple(Q.shape)}')
print(f'scores shape:  {tuple(scores.shape)}   (pairwise similarities)')
print(f'weights shape: {tuple(weights.shape)}  (rows sum to 1: {weights.sum(dim=-1).numpy().round(4)})')
print(f'output shape:  {tuple(output.shape)}')

# ── Visualise the attention matrix and one row's weight distribution ──────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

im = axes[0].imshow(weights.numpy(), cmap='viridis', vmin=0, vmax=weights.max().item())
axes[0].set_title('Attention weights  $\\mathrm{softmax}(QK^\\top/\\sqrt{d_k})$')
axes[0].set_xlabel('Key index $j$  (attended to)')
axes[0].set_ylabel('Query index $i$  (attending)')
axes[0].set_xticks(range(N)); axes[0].set_yticks(range(N))
plt.colorbar(im, ax=axes[0], fraction=0.046)

row = 3
axes[1].bar(range(N), weights[row].numpy(), color='steelblue', edgecolor='k')
axes[1].set_title(f'Row {row}: how token {row} distributes its attention')
axes[1].set_xlabel('Key index $j$')
axes[1].set_ylabel('Weight')
axes[1].set_xticks(range(N))
axes[1].axhline(1.0 / N, color='red', ls='--', lw=1,
                label=f'Uniform baseline = 1/N = {1.0/N:.3f}')
axes[1].legend()

plt.tight_layout(); plt.show()

### Think about it

- The diagonal of the attention matrix is often (but not always) bright. Why would a   token naturally attend to itself, and what would it mean if a row had a *near-zero*   diagonal entry?
- The projections $W_Q$, $W_K$, $W_V$ here are random, not trained. What kind of   structure would you expect the attention matrix to gain after training on a real task?
- The scaling factor $1/\sqrt{d_k}$ keeps the dot-product scores from growing with   dimension. What would happen to the softmax distribution if you dropped this scaling   and used $d_k = 1024$?
- A CNN with a $3\times3$ kernel mixes information across 9 neighbouring positions per   layer. How many positions does one self-attention layer mix across for $N = 8$?   For $N = 4096$?

---

## Part 2 — Attention on a galaxy image (the ViT trick)

Vision Transformers turn an image into a *sequence* by chopping it into non-overlapping
patches and flattening each one into a token. A $64 \times 64$ image with $8 \times 8$
patches gives $8 \times 8 = 64$ tokens. Each token is then linearly projected into the
model's embedding dimension and fed through self-attention exactly like Part 1.

Below: same `make_galaxy(64)` synthetic galaxy from Notebook 05. We split it into 64
patches, project each to a 16-D token, and inspect which patches attend to which.

In [ ]:
# ── Part 2: Patch embedding + attention on a synthetic galaxy ──────────────

def make_galaxy(size=64):
    """Same helper as Notebook 05: bulge + disc + spiral + noise."""
    x = np.linspace(-3, 3, size); y = np.linspace(-3, 3, size)
    X, Y = np.meshgrid(x, y)
    r = np.sqrt(X**2 + Y**2); theta = np.arctan2(Y, X)
    bulge  = np.exp(-r**2 / 0.4)
    disc   = 0.5 * np.exp(-r / 2.0)
    spiral = 0.35 * disc * np.maximum(0, np.cos(2 * theta - r * 1.5))
    rng = np.random.RandomState(42)
    img = bulge + disc + spiral + 0.04 * rng.randn(size, size)
    img = np.clip(img, 0, None); img /= img.max()
    return img

IMG_SIZE   = 64
PATCH      = 8
GRID       = IMG_SIZE // PATCH         # 8 patches per side
N_PATCHES  = GRID * GRID                # 64 tokens
D_PATCH    = PATCH * PATCH              # 64-D flat patch
D_MODEL    = 16                          # embedding dim per token

GALAXY = make_galaxy(IMG_SIZE)

# Chop into patches → (N_PATCHES, D_PATCH)
patches = (torch.tensor(GALAXY, dtype=torch.float32)
           .unfold(0, PATCH, PATCH).unfold(1, PATCH, PATCH)
           .contiguous().view(N_PATCHES, D_PATCH))

# Linear projection to embedding dim (random — untrained)
W_embed = torch.randn(D_PATCH, D_MODEL) / np.sqrt(D_PATCH)
tokens  = patches @ W_embed              # (64, 16)

# Self-attention exactly as in Part 1
W_Q = torch.randn(D_MODEL, D_MODEL) / np.sqrt(D_MODEL)
W_K = torch.randn(D_MODEL, D_MODEL) / np.sqrt(D_MODEL)
W_V = torch.randn(D_MODEL, D_MODEL) / np.sqrt(D_MODEL)

Q = tokens @ W_Q; K = tokens @ W_K; V = tokens @ W_V
attn = F.softmax(Q @ K.T / np.sqrt(D_MODEL), dim=-1)   # (64, 64)

# ── Visualise: galaxy with patch grid, full attention matrix,
#               and attention map for one chosen query patch ──────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(GALAXY, cmap='inferno')
for i in range(1, GRID):
    axes[0].axhline(i * PATCH - 0.5, color='white', lw=0.6, alpha=0.5)
    axes[0].axvline(i * PATCH - 0.5, color='white', lw=0.6, alpha=0.5)
# Mark the chosen query patch (centre of the galaxy)
query_idx = (GRID // 2) * GRID + (GRID // 2)        # patch at grid (4, 4)
qi, qj    = divmod(query_idx, GRID)
axes[0].add_patch(plt.Rectangle((qj * PATCH - 0.5, qi * PATCH - 0.5),
                                PATCH, PATCH, ec='cyan', fc='none', lw=2))
axes[0].set_title(f'Galaxy with 8×8 patch grid\nquery patch (cyan) = index {query_idx}')
axes[0].axis('off')

im = axes[1].imshow(attn.numpy(), cmap='viridis')
axes[1].set_title(f'Full attention matrix  ({N_PATCHES}×{N_PATCHES})')
axes[1].set_xlabel('Key patch'); axes[1].set_ylabel('Query patch')
plt.colorbar(im, ax=axes[1], fraction=0.046)

# Reshape the query's attention row back to the 8×8 grid — a spatial attention map
attn_map = attn[query_idx].view(GRID, GRID).numpy()
im2 = axes[2].imshow(attn_map, cmap='magma')
axes[2].set_title(f'Attention map for query patch {query_idx}\n(where it looks)')
axes[2].set_xticks(range(GRID)); axes[2].set_yticks(range(GRID))
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout(); plt.show()

print(f'patches tensor: {tuple(patches.shape)}  (N_patches, patch_pixels)')
print(f'tokens tensor:  {tuple(tokens.shape)}   (N_patches, d_model)')
print(f'attention:      {tuple(attn.shape)}   (rows sum to 1)')

### Think about it

- With random (untrained) projections the attention map looks essentially noise. After   training on a galaxy-classification task, what *spatial* structure would you expect   the centre-patch attention map to develop?
- The attention matrix is $64 \times 64$ for this image. If you doubled the image to   $128 \times 128$ with the same $8 \times 8$ patches, how large does the attention   matrix become? Why does this make ViTs expensive on high-resolution images?
- A CNN's first layer can only mix information within a $3 \times 3$ neighbourhood.   How many CNN layers would you need before a centre pixel could influence the   four corner pixels of a $64 \times 64$ image? Attention does it in one layer.
- Patch tokens carry no information about *where* in the image they came from —   permuting patches gives the same attention matrix (up to relabelling).   Why do ViTs add a learned positional embedding to each token?

---

## Exercise — Implement `scaled_dot_product_attention`

The whole transformer family rests on a single ten-line function. Implement it.
Given $Q$, $K$, $V$ each of shape $(N, d_k)$, return the attention output of shape
$(N, d_k)$ following:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

The test cell will compare your output to `torch.nn.functional.scaled_dot_product_attention`.

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    """
    Args:
        Q, K, V : torch.Tensor of shape (N, d_k)
    Returns:
        torch.Tensor of shape (N, d_k) — the attention output.
    """
    # Steps:
    #   1. scores  = Q @ K.T / sqrt(d_k)
    #   2. weights = softmax(scores, dim=-1)
    #   3. return weights @ V
    raise NotImplementedError('Fill in scaled_dot_product_attention')

# ── Test: compare to PyTorch's reference implementation ────────────────────
torch.manual_seed(123)
Q_test = torch.randn(8, 16)
K_test = torch.randn(8, 16)
V_test = torch.randn(8, 16)

out_yours = scaled_dot_product_attention(Q_test, K_test, V_test)

# PyTorch's reference expects a leading batch/head dimension
out_ref = F.scaled_dot_product_attention(
    Q_test.unsqueeze(0), K_test.unsqueeze(0), V_test.unsqueeze(0)
).squeeze(0)

assert out_yours.shape == out_ref.shape, f'shape mismatch: {out_yours.shape} vs {out_ref.shape}'
assert torch.allclose(out_yours, out_ref, atol=1e-5), 'values disagree with F.scaled_dot_product_attention'
print('✓ Matches torch.nn.functional.scaled_dot_product_attention.')

---

> Try the exercise yourself first. The solution below shows one clean implementation
> and connects it back to the parts above.

## Solution — `scaled_dot_product_attention`

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    d_k     = Q.shape[-1]
    scores  = Q @ K.transpose(-2, -1) / np.sqrt(d_k)   # (N, N) pairwise similarities
    weights = F.softmax(scores, dim=-1)                # rows sum to 1
    return weights @ V                                  # (N, d_k) weighted values

# ── What this connects back to ─────────────────────────────────────
# Part 1: the 8×8 weight matrix you visualised is exactly the `weights` here.
# Part 2: the 64×64 attention matrix on patch tokens uses this same function;
#         the only extra step is the patch-embedding linear projection.
# A real transformer adds: multi-head splitting, a residual stream, an MLP block,
# and learned positional embeddings — but the kernel of computation is this.

---

## Going further

### Part A — Go deeper

Real transformers use **multi-head** attention: instead of one $(d_k)$-dimensional
attention computation, they split the channels into $h$ heads of size $d_k / h$ each,
run attention $h$ times in parallel, then concatenate. Different heads learn to attend
to different relations (e.g. one head tracks bulge–disc separation, another tracks
spiral-arm orientation).

**Challenge:** Extend your `scaled_dot_product_attention` to multi-head form. Given
$X$ of shape $(N, d)$ with $d = 16$ and $h = 4$ heads, split into 4 streams of
$d_k = 4$, run attention on each, concatenate back to $(N, 16)$, and apply a final
output projection $W_O$. Compare the per-head attention maps from Part 2 — do they
look different on the galaxy?

### Part B — Lead forward

Attention gives a *deterministic* point estimate of which tokens matter, just as
Notebooks 01–05 gave point estimates of weights. But for scientific inference we
often want a **distribution** over possible answers, not a single most-likely value
— especially when data are noisy and models are imperfect.

**Challenge:** Before starting Notebook 06, write one sentence on what would change
if every weight in this notebook (including $W_Q$, $W_K$, $W_V$) were a *distribution*
rather than a single tensor. What new quantity could you then compute that you cannot
compute with point-estimate attention?

---

### References

| | |
|---|---|
| Primary | Vaswani et al. (2017) — *Attention Is All You Need*. NeurIPS. [arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762) |
| Primary | Dosovitskiy et al. (2020) — *An Image Is Worth 16x16 Words: Transformers for Image Recognition at Scale*. ICLR 2021. [arxiv.org/abs/2010.11929](https://arxiv.org/abs/2010.11929) |
| History | Bahdanau, Cho & Bengio (2014) — *Neural Machine Translation by Jointly Learning to Align and Translate*. ICLR 2015. [arxiv.org/abs/1409.0473](https://arxiv.org/abs/1409.0473) |
| Video | Karpathy — *Let's build GPT: from scratch, in code, spelled out*. [youtube.com/watch?v=kCc8FmEb1nY](https://www.youtube.com/watch?v=kCc8FmEb1nY) |